# PetMatch: treino v2 (EfficientNetV2 + detector de cachorro)

Segunda versão do modelo de 120 raças. Mudanças em relação à v1 (MobileNetV2):

- **Rede mais forte:** EfficientNetV2-B3 em 300×300, no lugar da MobileNetV2 em 224×224.
  Raças pequenas e parecidas (Chihuahua, Papillon, Lulu da Pomerânia) dependem de detalhes
  que a rede menor perdia.
- **Aumento de dados mais realista:** zoom para fora e deslocamento, para o modelo aprender
  com o cachorro pequeno no quadro, como em fotos reais.
- **Detector de cachorro:** um segundo modelo (MobileNetV2 do ImageNet) que diz se a foto
  tem um cachorro. A API usa para responder "não é um cachorro" em vez de chutar uma raça.

**Como usar:** *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*,
depois *Ambiente de execução → Executar tudo*. Leva cerca de **1h a 1h30** na T4.

No final você baixa três arquivos, que vão para `backend/modelo/`:

| Arquivo | Para que serve |
|---|---|
| `modelo_racas.onnx` | Modelo de raças (substitui o atual) |
| `classes.json` | Ordem exata das 120 raças (substitui o atual) |
| `modelo_cachorro.onnx` | Detector de cachorro (arquivo novo) |

A entrada continua sendo RGB com valores 0–255 e a normalização fica dentro do modelo.
A API lê o tamanho (300×300) direto do arquivo ONNX, então o código não precisa mudar.

## 1. Conferir a GPU

In [ ]:
import tensorflow as tf
import keras

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
assert tf.config.list_physical_devices("GPU"), "Sem GPU: Ambiente de execução → Alterar o tipo → GPU T4"

## 2. Conectar o Google Drive

O Colab apaga `/content` quando desconecta. O modelo treinado e os gráficos vão para o Drive.

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

PASTA_DRIVE = "/content/drive/MyDrive/pets_identificator_v2"
os.makedirs(PASTA_DRIVE, exist_ok=True)
print("Arquivos finais serão salvos em:", PASTA_DRIVE)

## 3. Baixar o Stanford Dogs (~750 MB, 120 raças, ~20 mil fotos)

Se o site de Stanford estiver fora do ar, a célula tenta o Kaggle automaticamente.

In [ ]:
import subprocess
from pathlib import Path

ORIGEM = Path("/content/Images")

if not ORIGEM.exists():
    ok = subprocess.run(
        "wget -q http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar -O /content/images.tar"
        " && tar -xf /content/images.tar -C /content/",
        shell=True,
    ).returncode == 0
    if not ok:
        print("Stanford fora do ar, baixando pelo Kaggle…")
        import kagglehub
        caminho = Path(kagglehub.dataset_download("jessicali9530/stanford-dogs-dataset"))
        ORIGEM = next(p for p in caminho.rglob("Images") if p.is_dir())

pastas = sorted(p for p in ORIGEM.iterdir() if p.is_dir())
print("Pasta:", ORIGEM)
print("Raças encontradas:", len(pastas))
print("Exemplo:", pastas[0].name)
assert len(pastas) == 120, "Esperava 120 pastas de raças"

## 4. Nomes das raças em português

As pastas vêm como `n02085620-Chihuahua`. Este dicionário converte o final do nome
para o nome mostrado no app. As 5 raças do projeto original mantêm o mesmo nome
(Labrador, Golden Retriever, Beagle, Pug, Chihuahua), então o histórico antigo continua batendo.

Pasta sem tradução usa o nome em inglês (com aviso), em vez de quebrar.

In [ ]:
NOMES_PT = {
    "Chihuahua": "Chihuahua",
    "Japanese_spaniel": "Chin Japonês",
    "Maltese_dog": "Maltês",
    "Pekinese": "Pequinês",
    "Shih-Tzu": "Shih Tzu",
    "Blenheim_spaniel": "Cavalier King Charles Spaniel",
    "papillon": "Papillon",
    "toy_terrier": "Toy Terrier",
    "Rhodesian_ridgeback": "Rhodesian Ridgeback",
    "Afghan_hound": "Galgo Afegão",
    "basset": "Basset Hound",
    "beagle": "Beagle",
    "bloodhound": "Bloodhound",
    "bluetick": "Bluetick Coonhound",
    "black-and-tan_coonhound": "Black and Tan Coonhound",
    "Walker_hound": "Treeing Walker Coonhound",
    "English_foxhound": "Foxhound Inglês",
    "redbone": "Redbone Coonhound",
    "borzoi": "Borzói",
    "Irish_wolfhound": "Lébrel Irlandês",
    "Italian_greyhound": "Galguinho Italiano",
    "whippet": "Whippet",
    "Ibizan_hound": "Podengo Ibicenco",
    "Norwegian_elkhound": "Elkhound Norueguês",
    "otterhound": "Otterhound",
    "Saluki": "Saluki",
    "Scottish_deerhound": "Deerhound",
    "Weimaraner": "Weimaraner",
    "Staffordshire_bullterrier": "Staffordshire Bull Terrier",
    "American_Staffordshire_terrier": "American Staffordshire Terrier",
    "Bedlington_terrier": "Bedlington Terrier",
    "Border_terrier": "Border Terrier",
    "Kerry_blue_terrier": "Kerry Blue Terrier",
    "Irish_terrier": "Terrier Irlandês",
    "Norfolk_terrier": "Norfolk Terrier",
    "Norwich_terrier": "Norwich Terrier",
    "Yorkshire_terrier": "Yorkshire",
    "wire-haired_fox_terrier": "Fox Terrier de Pelo Duro",
    "Lakeland_terrier": "Lakeland Terrier",
    "Sealyham_terrier": "Sealyham Terrier",
    "Airedale": "Airedale Terrier",
    "cairn": "Cairn Terrier",
    "Australian_terrier": "Terrier Australiano",
    "Dandie_Dinmont": "Dandie Dinmont Terrier",
    "Boston_bull": "Boston Terrier",
    "miniature_schnauzer": "Schnauzer Miniatura",
    "giant_schnauzer": "Schnauzer Gigante",
    "standard_schnauzer": "Schnauzer Médio",
    "Scotch_terrier": "Terrier Escocês",
    "Tibetan_terrier": "Terrier Tibetano",
    "silky_terrier": "Silky Terrier",
    "soft-coated_wheaten_terrier": "Soft-Coated Wheaten Terrier",
    "West_Highland_white_terrier": "West Highland White Terrier",
    "Lhasa": "Lhasa Apso",
    "flat-coated_retriever": "Flat-Coated Retriever",
    "curly-coated_retriever": "Curly-Coated Retriever",
    "golden_retriever": "Golden Retriever",
    "Labrador_retriever": "Labrador",
    "Chesapeake_Bay_retriever": "Chesapeake Bay Retriever",
    "German_short-haired_pointer": "Braco Alemão de Pelo Curto",
    "vizsla": "Vizsla",
    "English_setter": "Setter Inglês",
    "Irish_setter": "Setter Irlandês",
    "Gordon_setter": "Gordon Setter",
    "Brittany_spaniel": "Spaniel Bretão",
    "clumber": "Clumber Spaniel",
    "English_springer": "Springer Spaniel Inglês",
    "Welsh_springer_spaniel": "Springer Spaniel Galês",
    "cocker_spaniel": "Cocker Spaniel",
    "Sussex_spaniel": "Sussex Spaniel",
    "Irish_water_spaniel": "Spaniel de Água Irlandês",
    "kuvasz": "Kuvasz",
    "schipperke": "Schipperke",
    "groenendael": "Pastor Belga Groenendael",
    "malinois": "Pastor Belga Malinois",
    "briard": "Briard",
    "kelpie": "Kelpie Australiano",
    "komondor": "Komondor",
    "Old_English_sheepdog": "Old English Sheepdog",
    "Shetland_sheepdog": "Pastor de Shetland",
    "collie": "Collie",
    "Border_collie": "Border Collie",
    "Bouvier_des_Flandres": "Bouvier de Flandres",
    "Rottweiler": "Rottweiler",
    "German_shepherd": "Pastor Alemão",
    "Doberman": "Dobermann",
    "miniature_pinscher": "Pinscher Miniatura",
    "Greater_Swiss_Mountain_dog": "Grande Boiadeiro Suíço",
    "Bernese_mountain_dog": "Boiadeiro Bernês",
    "Appenzeller": "Boiadeiro de Appenzell",
    "EntleBucher": "Boiadeiro de Entlebuch",
    "boxer": "Boxer",
    "bull_mastiff": "Bulmastife",
    "Tibetan_mastiff": "Mastim Tibetano",
    "French_bulldog": "Buldogue Francês",
    "Great_Dane": "Dogue Alemão",
    "Saint_Bernard": "São Bernardo",
    "Eskimo_dog": "Cão Esquimó",
    "malamute": "Malamute do Alasca",
    "Siberian_husky": "Husky Siberiano",
    "affenpinscher": "Affenpinscher",
    "basenji": "Basenji",
    "pug": "Pug",
    "Leonberg": "Leonberger",
    "Newfoundland": "Terra-Nova",
    "Great_Pyrenees": "Cão dos Pirineus",
    "Samoyed": "Samoieda",
    "Pomeranian": "Lulu da Pomerânia",
    "chow": "Chow Chow",
    "keeshond": "Keeshond",
    "Brabancon_griffon": "Petit Brabançon",
    "Pembroke": "Welsh Corgi Pembroke",
    "Cardigan": "Welsh Corgi Cardigan",
    "toy_poodle": "Poodle Toy",
    "miniature_poodle": "Poodle Miniatura",
    "standard_poodle": "Poodle Standard",
    "Mexican_hairless": "Pelado Mexicano",
    "dingo": "Dingo",
    "dhole": "Dhole",
    "African_hunting_dog": "Cão Selvagem Africano",
}

def nome_da_raca(pasta):
    """'n02099712-Labrador_retriever' -> 'Labrador'"""
    sufixo = pasta.name.split("-", 1)[1]
    if sufixo not in NOMES_PT:
        print(f"Aviso: sem tradução para '{sufixo}', usando o nome em inglês")
        return sufixo.replace("_", " ").title()
    return NOMES_PT[sufixo]

MAPA = {pasta: nome_da_raca(pasta) for pasta in pastas}
assert len(set(MAPA.values())) == 120, "Dois nomes de raça iguais"
print(len(MAPA), "raças, por exemplo:", list(MAPA.values())[:5])

## 5. Separar treino, validação e teste

Cada raça é dividida em **70% treino / 15% validação / 15% teste**, com semente fixa (`42`).
O teste tem fotos que o modelo **nunca viu**; é a acurácia dele que vai no relatório.

In [ ]:
import random
import shutil

DESTINO = Path("/content/dataset")
random.seed(42)
if DESTINO.exists():
    shutil.rmtree(DESTINO)

contagem = {"train": 0, "val": 0, "test": 0}
for pasta, nome in MAPA.items():
    imagens = sorted(pasta.glob("*.jpg"))
    random.shuffle(imagens)
    n_treino = int(len(imagens) * 0.70)
    n_val = int(len(imagens) * 0.15)
    partes = {
        "train": imagens[:n_treino],
        "val": imagens[n_treino:n_treino + n_val],
        "test": imagens[n_treino + n_val:],
    }
    for parte, arquivos in partes.items():
        destino = DESTINO / parte / nome
        destino.mkdir(parents=True, exist_ok=True)
        for arquivo in arquivos:
            shutil.copy(arquivo, destino / arquivo.name)
        contagem[parte] += len(arquivos)

print(contagem)  # esperado: ~14.400 / ~3.000 / ~3.200

## 6. Carregar as imagens

A **ordem das classes** é a alfabética dos nomes das pastas, e é ela que vai para o
`classes.json`. O índice que o modelo devolve é a posição nessa lista: nunca reordene
o arquivo à mão.

In [ ]:
IMG = 300   # tamanho nativo da EfficientNetV2-B3
LOTE = 32
AUTOTUNE = tf.data.AUTOTUNE

def carregar(parte, embaralhar):
    return keras.utils.image_dataset_from_directory(
        DESTINO / parte,
        image_size=(IMG, IMG),
        batch_size=LOTE,
        label_mode="int",
        shuffle=embaralhar,
        seed=42,
    )

treino_ds = carregar("train", embaralhar=True)
val_ds = carregar("val", embaralhar=False)
teste_ds = carregar("test", embaralhar=False)

CLASSES = treino_ds.class_names
NUM_CLASSES = len(CLASSES)
print(NUM_CLASSES, "classes. Primeiras:", CLASSES[:5])
assert NUM_CLASSES == 120

## 7. Aumento de dados

Aplicado **só no treino** e **fora do modelo**, para o modelo exportado ficar limpo.
O zoom vai de -30% (afasta: cachorro menor no quadro, com mais fundo) a +20% (aproxima),
e o deslocamento tira o cachorro do centro, como acontece em fotos de celular.

In [ ]:
aumento = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.08),
    keras.layers.RandomZoom(height_factor=(-0.2, 0.3), fill_mode="reflect"),
    keras.layers.RandomTranslation(0.1, 0.1, fill_mode="reflect"),
    keras.layers.RandomContrast(0.2),
    keras.layers.RandomBrightness(0.15, value_range=(0, 255)),
], name="aumento")

treino_aug = (
    treino_ds
    .map(lambda x, y: (aumento(x, training=True), y), num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE)
)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
teste_ds = teste_ds.cache().prefetch(AUTOTUNE)

## 8. Montar o modelo (EfficientNetV2-B3 + camada de 120 saídas)

A EfficientNetV2 do Keras já traz a normalização dentro dela (`include_preprocessing=True`)
e espera valores de 0 a 255, o mesmo formato que a API envia.

In [ ]:
base = keras.applications.EfficientNetV2B3(
    input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet", include_preprocessing=True
)
base.trainable = False  # fase 1: base congelada

entrada = keras.Input(shape=(IMG, IMG, 3), name="imagem")
x = base(entrada, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.4)(x)
saida = keras.layers.Dense(NUM_CLASSES, activation="softmax", name="probabilidades")(x)

modelo = keras.Model(entrada, saida, name="classificador_racas")
modelo.summary()

## 9. Treino, fase 1 (só a camada nova)

O `EarlyStopping` para quando a validação deixa de melhorar e volta aos melhores pesos.
O `ModelCheckpoint` salva no Drive a cada melhora: se o Colab cair, nada se perde.

In [ ]:
parada = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
checkpoint = keras.callbacks.ModelCheckpoint(
    f"{PASTA_DRIVE}/checkpoint.keras", monitor="val_loss", save_best_only=True
)

modelo.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3")],
)
hist1 = modelo.fit(treino_aug, validation_data=val_ds, epochs=8, callbacks=[parada, checkpoint])

## 10. Treino, fase 2 (ajuste fino)

Libera as últimas 80 camadas da base com taxa de aprendizado bem baixa (`2e-5`).
Compare a `val_accuracy` daqui com a da v1 (~83%): é ela que mostra se a troca de rede valeu.

In [ ]:
base.trainable = True
for camada in base.layers[:-80]:
    camada.trainable = False

modelo.compile(
    optimizer=keras.optimizers.Adam(2e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3")],
)
hist2 = modelo.fit(treino_aug, validation_data=val_ds, epochs=8, callbacks=[parada, checkpoint])

## 11. Gráfico do treino (para o relatório)

In [ ]:
import matplotlib.pyplot as plt

def junta(chave):
    return hist1.history[chave] + hist2.history[chave]

inicio_fase2 = len(hist1.history["accuracy"])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(junta("accuracy"), label="treino"); ax1.plot(junta("val_accuracy"), label="validação")
ax1.set_title("Acurácia"); ax1.axvline(inicio_fase2 - 0.5, ls="--", c="gray")
ax2.plot(junta("loss"), label="treino"); ax2.plot(junta("val_loss"), label="validação")
ax2.set_title("Perda (loss)"); ax2.axvline(inicio_fase2 - 0.5, ls="--", c="gray")
for ax in (ax1, ax2):
    ax.set_xlabel("época"); ax.legend()
plt.savefig(f"{PASTA_DRIVE}/grafico_treino.png", dpi=120, bbox_inches="tight")
plt.show()

## 12. Avaliação no conjunto de teste

Com 120 classes a matriz de confusão completa fica ilegível, então a célula mostra
as **raças com pior acerto** e os **pares que o modelo mais confunde**.

> **Para o relatório:** o Stanford Dogs vem do ImageNet, o mesmo conjunto do pré-treino
> da EfficientNetV2, então a acurácia no teste tende a ser otimista. Teste também com fotos suas.

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

resultado = modelo.evaluate(teste_ds, return_dict=True)
print(f"\nAcurácia no teste: {resultado['accuracy']:.2%}   top-3: {resultado['top3']:.2%}\n")

y_real = np.concatenate([y.numpy() for _, y in teste_ds])
prob_teste = modelo.predict(teste_ds, verbose=0)
y_prev = prob_teste.argmax(1)

relatorio = classification_report(y_real, y_prev, target_names=CLASSES, digits=3, output_dict=True)
with open(f"{PASTA_DRIVE}/relatorio_teste.txt", "w", encoding="utf-8") as f:
    f.write(classification_report(y_real, y_prev, target_names=CLASSES, digits=3))

piores = sorted(CLASSES, key=lambda c: relatorio[c]["f1-score"])[:10]
print("10 raças com pior F1:")
for c in piores:
    print(f"  {c:32s} F1={relatorio[c]['f1-score']:.2f}")

print("\nRaças que erravam na v1:")
for c in ["Chihuahua", "Papillon", "Lulu da Pomerânia", "Labrador", "Golden Retriever", "Pastor Alemão"]:
    print(f"  {c:32s} F1={relatorio[c]['f1-score']:.2f}  acerto={relatorio[c]['recall']:.0%}")

matriz = confusion_matrix(y_real, y_prev)
np.fill_diagonal(matriz, 0)
print("\nPares mais confundidos (real -> previsto):")
for i in np.argsort(matriz, axis=None)[::-1][:10]:
    real, prev = np.unravel_index(i, matriz.shape)
    print(f"  {CLASSES[real]:32s} -> {CLASSES[prev]:32s} {matriz[real, prev]} fotos")

## 13. Escolher o limiar de confiança da API

Abaixo do limiar a API responde **"Não reconhecido"** em vez de chutar. Com 120 raças a
confiança fica mais espalhada, então os 60% usados com 5 raças barrariam muitos acertos.

A tabela mostra, para cada limiar: quantas fotos do teste passariam (**cobertura**) e,
entre essas, quantas o modelo acerta (**precisão**). Um bom ponto costuma ficar onde a
precisão passa de ~90% sem derrubar demais a cobertura. Anote o valor sugerido: ele
vai no `LIMIAR_CONFIANCA` do `backend/predictor.py`.

In [ ]:
confianca = prob_teste.max(1) * 100
acertou = y_prev == y_real

print(f"{'limiar':>7} {'cobertura':>10} {'precisão':>9}")
sugestao = None
for limiar in range(20, 75, 5):
    passa = confianca >= limiar
    precisao = acertou[passa].mean() if passa.any() else 0
    print(f"{limiar:>6}% {passa.mean():>10.1%} {precisao:>9.1%}")
    if sugestao is None and precisao >= 0.90:
        sugestao = limiar

print(f"\nLimiar sugerido: {sugestao}%  (primeiro com precisão ≥ 90%)")

## 14. Testar com fotos suas

Usa **o mesmo pré-processamento da API** (Pillow). Para não travar o *Executar tudo*,
fica desligado: mude `TESTAR_FOTOS` para `True` e rode esta célula sozinha depois.

In [ ]:
from google.colab import files
from PIL import Image, ImageOps

def preprocessar(caminho):
    """Igual ao predictor.py da API: RGB, IMG x IMG, float32 de 0 a 255."""
    with Image.open(caminho) as img:
        img = ImageOps.exif_transpose(img).convert("RGB")
        img = img.resize((IMG, IMG), Image.Resampling.BILINEAR)
        return np.asarray(img, dtype=np.float32)[np.newaxis, ...]

TESTAR_FOTOS = False

if TESTAR_FOTOS:
    for nome_arquivo in files.upload():
        prob = modelo.predict(preprocessar(nome_arquivo), verbose=0)[0]
        print(f"\n{nome_arquivo}")
        for i in np.argsort(prob)[::-1][:3]:
            print(f"  {CLASSES[i]:32s} {prob[i]:6.1%}")

## 15. Salvar o modelo Keras e as classes no Drive

Antes de exportar: se a exportação falhar, não precisa treinar de novo.

In [ ]:
import json

modelo.save(f"{PASTA_DRIVE}/modelo_racas.keras")
with open(f"{PASTA_DRIVE}/classes.json", "w", encoding="utf-8") as f:
    json.dump(CLASSES, f, ensure_ascii=False, indent=2)

print(len(json.load(open(f"{PASTA_DRIVE}/classes.json", encoding="utf-8"))), "classes salvas")

## 16. Exportar para ONNX

Exporta como *SavedModel* e converte com o `tf2onnx` num processo separado, para evitar
conflito de versões com o TensorFlow já carregado.

> **Se der erro de `protobuf`** ou o TensorFlow quebrar depois do `pip install`:
> *Ambiente de execução → Reiniciar sessão*, rode as células 1 e 2 e depois:
> `modelo = keras.models.load_model(f"{PASTA_DRIVE}/modelo_racas.keras"); IMG = 300`
> e continue daqui.

In [ ]:
modelo.export("/content/modelo_savedmodel")

!pip install -q tf2onnx onnxruntime
!python -m tf2onnx.convert --saved-model /content/modelo_savedmodel --output /content/modelo_racas.onnx --opset 13
!ls -lh /content/modelo_racas.onnx

## 17. Conferir se o ONNX dá o mesmo resultado que o Keras

In [ ]:
import onnxruntime as ort

sessao = ort.InferenceSession("/content/modelo_racas.onnx", providers=["CPUExecutionProvider"])
nome_entrada = sessao.get_inputs()[0].name
print("Entrada:", nome_entrada, sessao.get_inputs()[0].shape)
print("Saída:  ", sessao.get_outputs()[0].name, sessao.get_outputs()[0].shape)

imagens, _ = next(iter(teste_ds))
imagens = imagens.numpy().astype(np.float32)
prob_keras = modelo.predict(imagens, verbose=0)
prob_onnx = sessao.run(None, {nome_entrada: imagens})[0]

diferenca = np.abs(prob_keras - prob_onnx).max()
iguais = (prob_keras.argmax(1) == prob_onnx.argmax(1)).mean()
print(f"Maior diferença: {diferenca:.6f}   previsões iguais: {iguais:.0%}")
assert prob_onnx.shape[1] == 120, "O ONNX não tem 120 saídas"
assert diferenca < 1e-3 and iguais == 1.0, "ONNX diferente do Keras!"
print("OK: o modelo ONNX é equivalente ao Keras.")

## 18. Detector de cachorro

O modelo de raças sempre escolhe uma das 120 raças, mesmo numa foto de gato ou de vaca.
O detector resolve isso: é a MobileNetV2 original do ImageNet (sem retreino), que conhece
1000 categorias. As categorias 151 a 268 do ImageNet são raças de cachorro; a soma das
probabilidades delas é a **probabilidade de ter um cachorro na foto**.

In [ ]:
detector_base = keras.applications.MobileNetV2(weights="imagenet", include_top=True)

entrada_det = keras.Input(shape=(224, 224, 3), name="imagem")
x = keras.layers.Rescaling(1 / 127.5, offset=-1)(entrada_det)  # 0..255 -> -1..1
probs_imagenet = detector_base(x, training=False)
prob_cachorro = keras.layers.Lambda(
    lambda p: keras.ops.sum(p[:, 151:269], axis=1, keepdims=True), name="prob_cachorro"
)(probs_imagenet)
detector = keras.Model(entrada_det, prob_cachorro, name="detector_cachorro")

# Quantas fotos de cachorro do teste o detector reconhece como cachorro
pc = np.concatenate([
    detector.predict(tf.image.resize(x_lote, (224, 224)), verbose=0)[:, 0]
    for x_lote, _ in teste_ds
])
print(f"{'limiar':>7} {'cachorros aceitos':>18}")
for limiar in (0.2, 0.3, 0.4, 0.5):
    print(f"{limiar:>7.0%} {(pc >= limiar).mean():>18.1%}")
print("A API usa 30%: só bloqueia fotos em que o detector quase não vê cachorro.")

## 19. Exportar o detector para ONNX

In [ ]:
detector.export("/content/detector_savedmodel")
!python -m tf2onnx.convert --saved-model /content/detector_savedmodel --output /content/modelo_cachorro.onnx --opset 13
!ls -lh /content/modelo_cachorro.onnx

sessao_det = ort.InferenceSession("/content/modelo_cachorro.onnx", providers=["CPUExecutionProvider"])
lote = tf.image.resize(next(iter(teste_ds))[0], (224, 224)).numpy().astype(np.float32)
diferenca = np.abs(detector.predict(lote, verbose=0) - sessao_det.run(None, {sessao_det.get_inputs()[0].name: lote})[0]).max()
print(f"Maior diferença Keras x ONNX: {diferenca:.6f}")
assert diferenca < 1e-3, "Detector ONNX diferente do Keras!"
print("OK: detector ONNX equivalente ao Keras.")

## 20. Copiar para o Drive e baixar

Depois de baixar, coloque os três arquivos em `backend/modelo/`:

1. `modelo_racas.onnx` e `classes.json` substituem os atuais (mesmos nomes).
2. `modelo_cachorro.onnx` é novo: a API passa a usar sozinha quando o arquivo existe.
3. No `backend/predictor.py`, troque `LIMIAR_CONFIANCA` pelo limiar sugerido no passo 13.
4. Reinicie a API: no terminal deve aparecer `Modelo carregado: 120 classes` e
   `Detector de cachorro carregado`.

In [ ]:
for arquivo in ("modelo_racas.onnx", "modelo_cachorro.onnx"):
    shutil.copy(f"/content/{arquivo}", f"{PASTA_DRIVE}/{arquivo}")
!ls -lh "{PASTA_DRIVE}"

files.download("/content/modelo_racas.onnx")
files.download(f"{PASTA_DRIVE}/classes.json")
files.download("/content/modelo_cachorro.onnx")